# XR-1 (Xiaomi-Robotics-1-5B) post-training on ANY XR-1-format dataset — RunPod

Point `DATASET` at a Hugging Face dataset repo or a local directory holding episodes in
Xiaomi's JSON format (one JSON per episode + the videos it references — see
`Xiaomi-Robotics-1/xr1/docs/data_format.md`). The notebook computes the normalisation
statistics with XR-1's own formulas, holds out a validation split, checks one sample
through **their** loader, trains with **their** launcher, and pushes the checkpoint in
the exact layout their inference server loads.

Everything model-side is Xiaomi's released code, unmodified except for four env-gated,
marker-guarded patches (cell 2). The FR5 LeRobot → XR-1 conversion is an **opt-in** step
(`LEROBOT_SOURCE`); for a dataset that is already in XR-1 format leave it empty.

Lessons from the pi0 runs, baked in so they cannot recur:

| trap | guard |
|---|---|
| stale pod checkout / stale kernel modules | hard-sync to `origin/main`, drop imported modules, assert symbols |
| stale dataset copy | always re-sync; instruction-diversity report |
| `/dev/shm` 64 MB kills their hardcoded 8 workers | `file_system` sharing patched into `train.py`, workers configurable |
| wandb blocking on an interactive login | mode resolved in the params cell, never a prompt |
| losses invisible in offline mode | `CSVLogger` patched in; `metrics.csv` tailed during training and pushed |
| optimizer JIT needs `nvcc` the pod may lack | `nvcc` detected → `FusedAdam` or `torch.optim.AdamW` |
| format mismatch found hours in | one sample through their `JsonDataset` + `CustomCollate` before torchrun |

GPU: one **A100-80**. The full 5B fine-tune needs ~80 GB before activations, so the default
**freezes the VLM and trains the 0.6 B DiT + projectors** (`FREEZE_VLM = True`).

## 0 · Pod setup — sync our repo, verify it is current

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

WORK     = Path(os.environ.get("XR1_WORK", "/workspace"))
REPO_URL = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
REPO_DIR = WORK / "fairino-fr5-act-pipeline"
BRANCH   = "main"


def _git(*args, check=True):
    r = subprocess.run(["git", "-C", str(REPO_DIR), *args], capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()


if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"clone failed:\n{r.stderr}")
    print(f"cloned {REPO_URL}")
else:
    # `pull --ff-only` refuses on a dirty tree and a swallowed failure once left this
    # notebook running week-old code. Hard-sync, and say exactly what gets discarded.
    _git("fetch", "origin", BRANCH)
    local, remote = _git("rev-parse", "HEAD"), _git("rev-parse", f"origin/{BRANCH}")
    if local != remote:
        dirty = _git("status", "--porcelain")
        if dirty:
            print("discarding local changes in the pod checkout:")
            for line in dirty.splitlines()[:10]:
                print(f"    {line}")
            _git("reset", "--hard", "HEAD"); _git("clean", "-fd")
        _git("checkout", BRANCH); _git("reset", "--hard", f"origin/{BRANCH}")
        print(f"synced {local[:8]} -> {remote[:8]}")
    else:
        print(f"already at origin/{BRANCH}")

sys.path[:0] = [str(REPO_DIR / "common"), str(REPO_DIR / "xr1_eef")]
os.chdir(REPO_DIR)

# a git sync cannot touch modules this kernel already imported — drop them so the
# next `from xr1_stats import ...` re-reads from disk (order-independent)
importlib.invalidate_caches()
for _m in ("xr1_stats", "fr5_to_xr1", "dataset", "lerobot_patches", "proprio"):
    if sys.modules.pop(_m, None) is not None:
        print(f"  dropped stale {_m}")

import xr1_stats as _xs
for _need in ("Stats", "write_yaml", "find_jsons", "episode_relative_actions", "episode_states"):
    assert hasattr(_xs, _need), f"xr1_stats.{_need} missing -> stale checkout; restart the kernel and re-run"
print(f"\nHEAD  {_git('log', '--oneline', '-1')}")
print(f"clean {'yes' if not _git('status', '--porcelain') else 'NO'}")

## 1 · Parameters

`DATASET` is the only thing that must change per dataset. `MAX_STEPS`, `BATCH_SIZE`,
`FREEZE_VLM` are the usual knobs.

In [ ]:
import getpass, os
from pathlib import Path

# ── credentials ────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (hf_..., read for the data + write for the push): ").strip()
assert HF_TOKEN.startswith("hf_")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = str(WORK / "hf_cache")             # persistent volume, not the container overlay
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "") or ""    # <- or paste here; empty = offline, never a prompt

# ── the dataset (XR-1 JSON format) ─────────────────────────────────────────────
DATASET         = "Slifold/fr5-xr1"        # HF dataset repo id  OR  a local directory
DATASET_NAME    = "fr5"                    # becomes configs/data/<name>.yaml and the run name
LEROBOT_SOURCE  = ""                       # OPT-IN: a LeRobot-v2 FR5 set (repo id or dir) to convert
                                           # into XR-1 format first. Leave "" when DATASET already is.
VAL_FRAC, SEED  = 0.05, 42                 # held-out episodes (by sorted filename, seeded); never trained on

# ── training ───────────────────────────────────────────────────────────────────
XR1_MODEL_REPO  = "XiaomiRobotics/Xiaomi-Robotics-1-5B"
MAX_STEPS       = 5000     # their reference: 10 000 @ batch 48. Every SAVE_INTERVAL is a checkpoint.
BATCH_SIZE      = 48       # per GPU. If OOM with FREEZE_VLM: 24.
SAVE_INTERVAL   = 1000
FREEZE_VLM      = True     # True  = DiT + projectors (0.6 B) — fits one A100-80
                           # False = their full recipe; needs >=2-4 GPUs with ZeRO. Not one A100-80.
NUM_WORKERS     = 4        # their code hardcodes 8

# ── paths ──────────────────────────────────────────────────────────────────────
XR1_DIR   = WORK / "Xiaomi-Robotics-1"
XR1_PKG   = XR1_DIR / "xr1"
RAW_DIR   = WORK / "datasets" / DATASET_NAME          # the dataset as downloaded / converted
TRAIN_DIR = WORK / "xr1_train" / DATASET_NAME          # json/ with absolute video paths + val_episodes.json
CKPT_PT   = XR1_PKG / "pretrained_ckpt" / "model_states.pt"
PROJECT, EXP = "xr1", DATASET_NAME
OUT_DIR   = XR1_PKG / "outputs" / f"project_{PROJECT}" / EXP
PUSH_REPO = "auto"                                     # auto -> <you>/<DATASET_NAME>-xr1-5b

os.environ["WANDB_MODE"] = "online" if WANDB_API_KEY else "offline"
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print(f"WANDB: ONLINE (key ...{WANDB_API_KEY[-4:]})")
else:
    print("WANDB: OFFLINE — losses are still in metrics.csv (CSVLogger patch); sync later with `wandb sync`")
print(f"dataset {DATASET!r} -> {RAW_DIR}   {'(converted from ' + LEROBOT_SOURCE + ')' if LEROBOT_SOURCE else ''}")
print(f"steps {MAX_STEPS}  batch {BATCH_SIZE}  freeze_vlm {FREEZE_VLM}  workers {NUM_WORKERS}  ->  {OUT_DIR}")

## 2 · Environment — their pinned stack, their repo, our four patches

`torch 2.8.0 / transformers 4.57.1 / deepspeed 0.18.9 / lightning 2.5.3 / decord` — this
**replaces** the pod's torch. Flash-attention 2 is required by their VLM build.

In [ ]:
import shutil, subprocess, sys, platform
print("python", platform.python_version(), "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
      capture_output=True, text=True).stdout.strip())

def sh(cmd, cwd=None, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True)
    if check and r.returncode:
        raise SystemExit(f"command failed ({r.returncode}): {cmd}")

if not XR1_DIR.exists():
    sh(f"git clone --depth 1 https://github.com/XiaomiRobotics/Xiaomi-Robotics-1 {XR1_DIR}")
else:
    sh("git fetch -q origin && git reset -q --hard origin/main", cwd=XR1_DIR)
    print("XR-1 repo reset to origin/main (our patches are re-applied below)")

sh(f"{sys.executable} -m pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128")
sh(f"{sys.executable} -m pip install -q -e {XR1_PKG}")            # assets/requirements.txt (transformers==4.57.1, decord 0.6.0 py3 wheel, ...)
sh(f"{sys.executable} -m pip install -q ninja")
_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
_wheel = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
          f"flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-{_py}-{_py}-linux_x86_64.whl")   # cp39..cp313 exist
sh(f"{sys.executable} -m pip install -q {_wheel} || {sys.executable} -m pip install -q flash-attn==2.8.3 --no-build-isolation")
sh("apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libgles2 tmux > /dev/null 2>&1 || true", check=False)

# deepspeed's FusedAdam is JIT-compiled with nvcc on first use; without nvcc the run
# dies at optimizer build. Their build_optimizer takes any "module.Class", so fall
# back to torch.optim.AdamW (same betas/eps/wd; DeepSpeed ZeRO wraps it fine).
NVCC = shutil.which("nvcc") or (Path("/usr/local/cuda/bin/nvcc").exists() and "/usr/local/cuda/bin/nvcc")
OPTIMIZER = "deepspeed.ops.adam.FusedAdam" if NVCC else "torch.optim.AdamW"
print(f"\nnvcc: {NVCC or 'NOT FOUND'} -> optimizer {OPTIMIZER}")

In [ ]:
# Import check in a FRESH interpreter (this kernel may hold pre-upgrade modules).
import subprocess, sys
chk = subprocess.run([sys.executable, "-c", """
import torch, transformers, deepspeed, lightning, decord, flash_attn, mibot
from mibot.utils.io import ACTION_DIM, STATE_DIM, recover_action
print('torch', torch.__version__, '| transformers', transformers.__version__, '| deepspeed', deepspeed.__version__,
      '| lightning', lightning.__version__, '| flash_attn', flash_attn.__version__, '| decord', decord.__version__)
assert transformers.__version__ == '4.57.1', transformers.__version__
assert torch.cuda.is_available()
print('mibot importable, ACTION_DIM', ACTION_DIM, 'STATE_DIM', STATE_DIM)
"""], capture_output=True, text=True, cwd=str(XR1_PKG))
print(chk.stdout)
if chk.returncode:
    raise SystemExit(chk.stderr[-2000:])

### Patches to their code (idempotent, marker-guarded)

1. `tools/train.py` — `set_sharing_strategy("file_system")`: RunPod caps `/dev/shm` at 64 MB.
2. `tools/train.py` — a `CSVLogger` next to their `WandbLogger`, so losses exist on disk.
3. `base_datamodule.py` — `num_workers` from `XR1_NUM_WORKERS` (hardcoded 8).
4. `XR1.py` — `XR1_FREEZE_VLM=1` freezes the VLM; `json_dataset.py` — `lru_cache(32)` → 512.

In [ ]:
import ast
from pathlib import Path

def patch(path, old, new, marker):
    p = Path(path); s = p.read_text()
    if marker in s:
        print(f"  already patched: {p.relative_to(XR1_DIR)}"); return
    assert old in s, f"anchor not found in {p} — their code changed; inspect before continuing:\n{old}"
    p.write_text(s.replace(old, new, 1)); print(f"  patched: {p.relative_to(XR1_DIR)}")

patch(XR1_PKG / "tools/train.py",
      "import hydra\n",
      "import hydra\nimport torch.multiprocessing  # fr5-patch: /dev/shm is 64 MB on RunPod\n"
      "torch.multiprocessing.set_sharing_strategy(\"file_system\")\n",
      "fr5-patch: /dev/shm")
patch(XR1_PKG / "tools/train.py",
      "    logging.getLogger(\"lightning.pytorch\").setLevel(logging.INFO)\n",
      "    from lightning.pytorch.loggers import CSVLogger  # fr5-patch: losses on disk in offline mode\n"
      "    logger.append(CSVLogger(save_dir=cfg.trainer.default_root_dir, name=\"csv_logs\"))\n"
      "    logging.getLogger(\"lightning.pytorch\").setLevel(logging.INFO)\n",
      "fr5-patch: losses on disk")
patch(XR1_PKG / "mibot/data/datamodule/base_datamodule.py",
      "            num_workers=8,",
      "            num_workers=int(__import__('os').environ.get('XR1_NUM_WORKERS', 8)),  # fr5-patch",
      "XR1_NUM_WORKERS")
patch(XR1_PKG / "mibot/models/VLA/XR1.py",
      "        self._build_model()\n",
      "        self._build_model()\n"
      "        if __import__('os').environ.get('XR1_FREEZE_VLM') == '1':  # fr5-patch: one A100-80 cannot hold the 5B full FT\n"
      "            self.vlm.requires_grad_(False)\n"
      "            print('[fr5-patch] VLM frozen; training DiT + projectors only')\n",
      "XR1_FREEZE_VLM")
patch(XR1_PKG / "mibot/data/datasets/json_dataset.py",
      "    @lru_cache(maxsize=32)",
      "    @lru_cache(maxsize=512)  # fr5-patch: hundreds of episode files",
      "fr5-patch: hundreds")

for f in ("tools/train.py", "mibot/data/datamodule/base_datamodule.py", "mibot/models/VLA/XR1.py", "mibot/data/datasets/json_dataset.py"):
    ast.parse((XR1_PKG / f).read_text())
print("all patched files parse")

## 3 · Data

Downloads (or uses) the dataset, optionally converting a LeRobot-v2 FR5 set first, then
builds `TRAIN_DIR/json` with **absolute** video paths (their loader resolves relative paths
from the directory training is launched in, which is not where the data lives) and a
seeded held-out split. Nothing in the held-out set is trained on.

In [ ]:
import json, random, shutil
from huggingface_hub import snapshot_download, hf_hub_download
import xr1_stats as xs

RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
if LEROBOT_SOURCE:                                       # opt-in: LeRobot-v2 FR5 -> XR-1 format
    import fr5_to_xr1 as fx
    src = Path(LEROBOT_SOURCE)
    if not src.exists():
        src = WORK / "datasets" / (LEROBOT_SOURCE.split("/")[-1] + "_lerobot")
        snapshot_download(LEROBOT_SOURCE, repo_type="dataset", local_dir=str(src), token=HF_TOKEN)   # always sync
    if RAW_DIR.exists(): shutil.rmtree(RAW_DIR)
    fx.main([str(src), "--out", str(RAW_DIR)])           # RAW_DIR/json/*.json with absolute video paths
elif Path(DATASET).exists():
    RAW_DIR = Path(DATASET)
else:
    snapshot_download(DATASET, repo_type="dataset", local_dir=str(RAW_DIR), token=HF_TOKEN)          # always sync

jsons = xs.find_jsons([RAW_DIR])
assert jsons, f"no episode JSON files under {RAW_DIR}"
print(f"{len(jsons)} episode JSONs under {RAW_DIR}")

# held-out split by sorted filename, seeded — reproducible for any dataset
rng = random.Random(SEED); order = sorted(jsons); rng.shuffle(order)
n_val = max(1, int(VAL_FRAC * len(order)))
val_files, train_files = sorted(order[:n_val]), sorted(order[n_val:])

# rewrite video paths to absolute, write the training copy
if TRAIN_DIR.exists(): shutil.rmtree(TRAIN_DIR)
(TRAIN_DIR / "json").mkdir(parents=True)
instructions, n_frames, cams = {}, 0, None
for f in train_files:
    ep = json.loads(f.read_text())
    for view in ep["observations"].values():
        for v in view:
            p = Path(v["path"])
            v["path"] = str(p if p.is_absolute() else (f.parent / p).resolve() if (f.parent / p).exists() else (RAW_DIR / p).resolve())
            assert Path(v["path"]).exists(), f"video not found: {v['path']} (referenced by {f.name})"
    (TRAIN_DIR / "json" / f.name).write_text(json.dumps(ep))
    n_frames += int(ep["num_frames"])
    g = ep["instruction"]["general"][0]
    cams = cams or g["images"]
    txt = g["conversations"][0]["value"].rsplit("\n", 1)[-1]
    instructions[txt] = instructions.get(txt, 0) + 1
(TRAIN_DIR / "val_episodes.json").write_text(json.dumps([str(f) for f in val_files]))

print(f"train {len(train_files)} episodes / {n_frames:,} frames   held out {len(val_files)} -> {TRAIN_DIR/'val_episodes.json'}")
print(f"views per sample: {cams}")
print(f"instructions: {len(instructions)} distinct over {len(train_files)} episodes")
for t, n in sorted(instructions.items(), key=lambda kv: -kv[1])[:12]: print(f"  {n:5d}  {t}")
if len(instructions) > 0.5 * len(train_files):
    print("  WARNING: nearly one instruction per episode — the model can use the text as an episode ID "
          "instead of a task description (this is what tools/rebalance_instructions.py fixed for FR5)")

# the referenced videos must open with THEIR decoder (decord)
import decord
ep = json.loads((TRAIN_DIR / "json" / train_files[0].name).read_text())
for view, infos in ep["observations"].items():
    vr = decord.VideoReader(infos[0]["path"], num_threads=2)
    assert len(vr) >= ep["num_frames"] + int(infos[0].get("start", 0)), f"{view}: {len(vr)} frames < num_frames"
    print(f"  decord {view}: {len(vr)} frames, {tuple(vr[0].shape)}")

# XR-1 5B weights (10.2 GB) -> where their README expects them (symlink, not a copy)
CKPT_PT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT_PT.exists():
    CKPT_PT.symlink_to(hf_hub_download(XR1_MODEL_REPO, "model_states.pt", token=HF_TOKEN))
print(f"weights: {CKPT_PT} -> {CKPT_PT.resolve().stat().st_size/1e9:.1f} GB")

## 4 · Normalisation statistics — XR-1's formulas, on the training split only

In [ ]:
import xr1_stats as xs
stats = xs.Stats()
for f in sorted((TRAIN_DIR / "json").glob("*.json")):
    stats.add(json.loads(f.read_text()))
mean, std, q01, q99 = stats.result()
DATA_CFG = XR1_PKG / "configs" / "data" / f"{DATASET_NAME}.yaml"
xs.write_yaml(DATA_CFG, TRAIN_DIR / "json", mean, std, q01, q99, batch_size=BATCH_SIZE)
active = [k for k, s in xs.PARTS.items() if std[:, s].max() > 0]
print(f"{stats.n:,} chunk starts   active action parts: {active}")
for k in active:
    s = xs.PARTS[k]; print(f"  {k:14s} std entry 0 {std[0, s].round(5)}   entry {xs.ACTION_LENGTH-1} {std[-1, s].round(4)}")
print(f"  state dims with a range: {int((q99 > q01).sum())} / 60")
assert std[:, xs.PARTS['left_ee_pos']].max() > 0 or std[:, xs.PARTS['right_ee_pos']].max() > 0, "no arm motion in the actions?"
print(f"wrote {DATA_CFG}")

## 5 · One sample through THEIR loader before torchrun

Exactly what training will use. Also measures tokens per sample so `MAX_LENGTH` (their
per-batch token budget — samples beyond it are silently **dropped**) is set from data.

In [ ]:
import os, sys, yaml, torch, numpy as np
os.environ["XR1_NUM_WORKERS"] = str(NUM_WORKERS)
sys.path.insert(0, str(XR1_PKG))
from mibot.data.datasets.json_dataset import JsonDataset
from mibot.data.collate.custom_collate import CustomCollate

params = yaml.safe_load(DATA_CFG.read_text())["data"]["params"]
params["max_steps"] = 1
ds = JsonDataset(params)
smp = ds[0]
assert tuple(smp["action"].shape) == (30, 60) and tuple(smp["state"].shape) == (1, 60), (smp["action"].shape, smp["state"].shape)
imgs = [c["image"] for m in smp["messages"] for c in m["content"] if isinstance(c, dict) and c.get("type") == "image"]
print(f"sample 0: action {tuple(smp['action'].shape)}, state {tuple(smp['state'].shape)}, images {len(imgs)} {[im.size for im in imgs]}")
a, s = smp["action"].numpy(), smp["state"].numpy()
print(f"  normalised action entry 0: {a[0, :7].round(2)}   |max| over chunk {np.abs(a).max():.2f}")
print(f"  normalised state: joints {s[0, :7].round(2)}  grip {s[0, 7]:.2f}   all in [-1, 1]: {bool((np.abs(s) <= 1).all())}")
assert np.isfinite(a).all() and np.abs(a).max() < 50, "normalised actions blew up — check std for near-zero dims"

col = CustomCollate()
batch = col([ds[0], ds[len(ds) // 2]])
tok = batch["input_ids"].shape[1] // 2
MAX_LENGTH = int(tok * BATCH_SIZE * 1.3)
print(f"  tokens per sample ~{tok} -> MAX_LENGTH {MAX_LENGTH} (their default 20000; below tokens x batch the collate silently DROPS samples)")
assert "pixel_values" in batch and tuple(batch["action"].shape) == (2, 30, 60)
print("loader OK")

## 6 · Train

Their launcher, unmodified, from `xr1/`. Progress is throttled to one line per ~30 s;
losses come from `metrics.csv` (the CSVLogger patch) and are printed as they land.
Interrupting the cell stops torchrun; the last `SAVE_INTERVAL` checkpoint survives.

There is **no validation loss** in their pipeline — whether the policy works is decided
offline afterwards, not by the curve.

In [ ]:
import os, re, signal, subprocess, sys, time
from pathlib import Path

env = dict(os.environ,
    RESOURCE_GPU="1", MAX_LENGTH=str(MAX_LENGTH), XR1_NUM_WORKERS=str(NUM_WORKERS),
    XR1_FREEZE_VLM="1" if FREEZE_VLM else "0", TOKENIZERS_PARALLELISM="false",
    WANDB_MODE=os.environ["WANDB_MODE"], HF_HOME=os.environ["HF_HOME"], HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED="1")
cmd = ["bash", "scripts/train.sh",
       f"trainer.project={PROJECT}", f"trainer.exp_name={EXP}", "trainer.default_root_dir=outputs",
       f"data={DATASET_NAME}", "model=posttrain", f"model.params.pretrained={CKPT_PT}",
       f"trainer.max_steps={MAX_STEPS}", f"trainer.save_interval={SAVE_INTERVAL}",
       f"trainer.optimizer.type={OPTIMIZER}"]
print("cwd", XR1_PKG); print(" ".join(cmd)); print()

LOG = XR1_PKG / f"train_{EXP}.log"
CSV = OUT_DIR / "csv_logs"
proc = subprocess.Popen(cmd, cwd=str(XR1_PKG), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
t0 = last_prog = 0.0; t0 = time.time(); seen_rows = 0; buf = b""

def tail_metrics():
    """Print new rows of lightning's metrics.csv (written every 50 steps)."""
    global seen_rows
    files = sorted(CSV.glob("version_*/metrics.csv"))
    if not files: return
    rows = files[-1].read_text().splitlines()
    if len(rows) <= 1: return
    head = rows[0].split(",")
    for r in rows[1 + seen_rows:]:
        d = dict(zip(head, r.split(",")))
        step = d.get("step", "?"); losses = {k.split("/")[-1]: v for k, v in d.items() if k.startswith("train/") and v}
        if losses: print(f"  step {step:>6s}  " + "  ".join(f"{k}={float(v):.4f}" for k, v in losses.items()) + f"  lr={float(d.get('lr') or 0):.2e}")
    seen_rows = len(rows) - 1

with open(LOG, "ab") as log:
    try:
        while True:
            chunk = proc.stdout.read1(65536) if hasattr(proc.stdout, "read1") else proc.stdout.read(4096)
            if not chunk:
                break
            log.write(chunk); buf += chunk
            *lines, buf = re.split(rb"[\r\n]", buf)              # progress bars use \r
            for raw in lines:
                line = raw.decode("utf-8", "replace").strip()
                if not line: continue
                if re.search(r"\d+/\d+ \[", line):                # lightning progress bar
                    if time.time() - last_prog > 30:
                        print("  " + line[:160]); last_prog = time.time(); tail_metrics()
                elif any(k in line for k in ("fr5-patch", "Trainable", "Non-trainable", "Saving", "Error", "error", "Traceback",
                                             "out of memory", "Loaded pretrained", "does not match")):
                    print(line[:200])
    except KeyboardInterrupt:
        print("\ninterrupting torchrun ..."); proc.send_signal(signal.SIGINT)
    proc.wait()
tail_metrics()
print(f"\nexit {proc.returncode} after {(time.time() - t0) / 3600:.2f} h  |  log: {LOG}")
print("checkpoints:", [c.name for c in sorted(OUT_DIR.glob('*.ckpt'))] or "NONE")

## 7 · Push

Uploads exactly the layout `mibot/server/deploy.py` loads — `<dir>/config.py` and
`<dir>/last.ckpt/checkpoint/mp_rank_00_model_states.pt` (weights only; DeepSpeed
optimizer shards are skipped) — plus the data config, metrics, log and held-out list.

```bash
hf download <repo> --local-dir posttrain            # on the inference machine
cd Xiaomi-Robotics-1/xr1 && bash scripts/deploy.sh $PWD/../posttrain 1 1
```

In [ ]:
from huggingface_hub import HfApi, whoami
api = HfApi(token=HF_TOKEN)
repo = PUSH_REPO if PUSH_REPO != "auto" else f"{whoami(token=HF_TOKEN)['name']}/{DATASET_NAME}-xr1-5b"
api.create_repo(repo, private=True, exist_ok=True)

weights = OUT_DIR / "last.ckpt" / "checkpoint" / "mp_rank_00_model_states.pt"
assert weights.exists(), f"no checkpoint at {weights} — did training reach SAVE_INTERVAL?"
metrics = sorted((OUT_DIR / "csv_logs").glob("version_*/metrics.csv"))
files = {"config.py": OUT_DIR / "config.py", "config.yaml": OUT_DIR / "config.yaml",
         "last.ckpt/checkpoint/mp_rank_00_model_states.pt": weights,
         f"{DATASET_NAME}.yaml": DATA_CFG, "train.log": LOG, "val_episodes.json": TRAIN_DIR / "val_episodes.json"}
if metrics: files["metrics.csv"] = metrics[-1]
card = f"""---
license: apache-2.0
tags: [robotics, vla, xiaomi-robotics-1]
---
# {repo.split('/')[-1]} — XR-1-5B post-trained on `{DATASET}`

- base `{XR1_MODEL_REPO}` · freeze_vlm={FREEZE_VLM} · batch {BATCH_SIZE} · {MAX_STEPS} steps · optimizer {OPTIMIZER} · lr 2e-5→5e-6 cosine (their recipe)
- action space: XR-1 native (relative Δpose in the current tool frame, 30 entries); views {cams}
- {len(train_files)} train episodes; held-out list in `val_episodes.json` (never trained on)
- serve: `bash scripts/deploy.sh <this dir> 1 1`
"""
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
for dst, src in files.items():
    if src.exists():
        api.upload_file(path_or_fileobj=str(src), path_in_repo=dst, repo_id=repo, commit_message=dst)
        print(f"  pushed {dst} ({src.stat().st_size/1e9:.2f} GB)")
print("done ->", f"https://huggingface.co/{repo}")

## 8 · Not covered

- **No validation loss exists in their pipeline.** The held-out episodes are listed in
  `val_episodes.json`; offline evaluation with their server in the loop (first-step error
  vs the don't-move baseline, gripper commitment — the `delta_joint/gate.py` criteria) is
  the next piece of work.
- **FR5 only:** the tool-frame orientation (`xr1_eef/README.md` §6 risk 1) is unverified,
  and `deploy_fr5_xr1.py` has not driven the arm.